In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# --- Пути для хранения ---
RAW_PATH = "data/raw/medical_staff.xlsx"
PROCESSED_PATH = "data/processed/medical_staff_cleaned.csv"
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# --- Скачивание файла ---
URL = "https://77.rosstat.gov.ru/storage/mediabank/%D0%A7%D0%B8%D1%81%D0%BB%D0%B5%D0%BD%D0%BD%D0%BE%D1%81%D1%82%D1%8C%20%D0%BC%D0%B5%D0%B4%D0%B8%D1%86%D0%B8%D0%BD%D1%81%D0%BA%D0%B8%D1%85%20%D0%BA%D0%B0%D0%B4%D1%80%D0%BE%D0%B2%20%D0%B7%D0%B0%202020-2024%20%D0%B3%D0%B3(1).xlsx"  # вставьте реальный URL с Мосстата
r = requests.get(URL, verify=False)
r.raise_for_status()
with open(RAW_PATH, "wb") as f:
    f.write(r.content)
print("Файл скачан")

In [ ]:
import os

# Создаем папку для обработанных данных
os.makedirs("data/processed", exist_ok=True)

# Сохраняем абсолютную численность
df_long_total.to_csv("data/processed/medical_staff_total.csv", index=False)

# Сохраняем численность на 10 000 населения
df_long_per_10k.to_csv("data/processed/medical_staff_per_10k.csv", index=False)

# Сохраняем исходный DataFrame с исходными колонками
df.to_csv("data/processed/medical_staff_cleaned.csv", index=False)

In [ ]:
# --- Чтение Excel ---
df = pd.read_excel(RAW_PATH, skiprows=2)
df

In [ ]:
# --- Убираем пустые строки (NaN везде) ---
df = df.dropna(how='all')
df

In [ ]:
df = df.iloc[1:-1].copy()
df

In [ ]:
# --- Переименовываем колонки ---
df = df.rename(columns={
    "Unnamed: 2": "Численность врачей на 10 000 человек населения",
    "Unnamed: 4": "Численность среднего медицинского персонала на 10 000 человек населения"
})
df

In [ ]:
# --- Проверяем текущие имена колонок ---
print(df.columns.tolist())

# --- Убираем лишние пробелы в названиях колонок ---
df.columns = df.columns.str.strip()

# --- Проверяем снова ---
print(df.columns.tolist())

In [ ]:
# --- Преобразование в длинный формат для абсолютной численности ---
df_long_total = df.melt(
    id_vars=["Годы"],
    value_vars=["Численность врачей", "Численность среднего медицинского персонала"],
    var_name="staff_type",
    value_name="count"
)

In [ ]:
# --- График абсолютной численности ---
plt.figure(figsize=(10,5))
sns.lineplot(data=df_long_total, x="Годы", y="count", hue="staff_type", marker='o')
plt.title("Численность медицинских кадров в Москве (всего человек)")
plt.xlabel("Год")
plt.ylabel("Количество человек")
plt.grid(True)

In [ ]:
# --- Преобразование в длинный формат для численности на 10 000 человек населения ---
df_long_per10k = df.melt(
    id_vars=["Годы"],
    value_vars=["Численность врачей на 10 000 человек населения", "Численность среднего медицинского персонала на 10 000 человек населения"],
    var_name="staff_type",
    value_name="count"
)


In [ ]:
# --- График на 10 000 человек населения ---
plt.figure(figsize=(10,5))
sns.lineplot(data=df_long_per10k, x="Годы", y="count", hue="staff_type", marker='o')
plt.title("Численность медицинских кадров в Москве (на 10 000 человек населения)")
plt.xlabel("Год")
plt.ylabel("Количество человек на 10 000 населения")
plt.grid(True)
plt.show()

In [ ]:
# --- Годовой прирост ---
df_growth = df[['Годы', 'Численность врачей', 'Численность среднего медицинского персонала']].copy()
df_growth['Врачи_рост_%'] = df_growth['Численность врачей'].pct_change() * 100
df_growth['СМП_рост_%'] = df_growth['Численность среднего медицинского персонала'].pct_change() * 100

print(df_growth)